# LeetCode 266 — Palindrome Permutation bằng Python

**Link đề:** https://leetcode.com/problems/palindrome-permutation/
**Độ khó:** Easy · **Chủ đề:** Hash Table, String, Bit Manipulation

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
class Solution:
    def canPermutePalindrome(self, s: str) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu **palindrome** (chuỗi đối xứng) và **hoán vị** (permutation) của chuỗi.
2. Thấy vì sao thử mọi hoán vị là bất khả thi (`n!`), từ đó tìm ra **điều kiện về tần suất**.
3. Rút ra insight: *"tối đa một ký tự xuất hiện số lẻ lần"*.
4. Cài đặt bằng `dict`/`Counter`, bằng `set` "bật/tắt", và bằng **bitmask XOR** 26 bit.
5. So sánh 5 cách và tự giải các biến thể (Palindrome Permutation II, Longest Palindrome).

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO chỉ định nghĩa hàm, không tự chạy.

## 1. Đề bài

Cho chuỗi `s`. Trả `True` nếu **có một hoán vị** (cách sắp xếp lại các ký tự) của `s` tạo thành palindrome, ngược lại `False`.

| `s` | Kết quả | Giải thích |
|---|:---:|---|
| `"code"` | `False` | 4 ký tự khác nhau, không cách xếp nào đối xứng |
| `"aab"` | `True` | xếp thành `"aba"` |
| `"carerac"` | `True` | xếp thành `"racecar"` |

**Constraints (theo hiểu biết về đề):** `1 <= s.length <= 5000`; `s` chỉ gồm chữ cái tiếng Anh **thường**.

Chú ý: ta **không cần tạo** palindrome, chỉ cần trả lời có/không.

In [1]:
examples = [("code", False), ("aab", True), ("carerac", True)]
for s, expected in examples:
    print(f"{s!r:>10} -> mong đợi {expected}")
print("'racecar' đọc ngược:", "racecar"[::-1])

    'code' -> mong đợi False
     'aab' -> mong đợi True
 'carerac' -> mong đợi True
'racecar' đọc ngược: racecar


## 2. Nền tảng Python

### 2.1. Palindrome và slicing `s[::-1]`

`s[::-1]` tạo chuỗi đảo ngược (`step = -1` đi từ cuối về đầu). Chuỗi là palindrome ⇔ `s == s[::-1]`.

In [2]:
for s in ["aba", "abba", "ab", "a"]:
    print(f"{s!r}: đảo = {s[::-1]!r}, palindrome? {s == s[::-1]}")

'aba': đảo = 'aba', palindrome? True
'abba': đảo = 'abba', palindrome? True
'ab': đảo = 'ba', palindrome? False
'a': đảo = 'a', palindrome? True


### 2.2. Hoán vị và `itertools.permutations`

Hoán vị của `"aab"` là mọi cách sắp xếp 3 ký tự. `itertools.permutations(s)` sinh ra từng hoán vị dưới dạng **tuple** ký tự; `"".join(t)` ghép tuple thành chuỗi.

Số hoán vị của chuỗi dài `n` là `n!` (tính cả trùng lặp) — tăng **cực nhanh**.

In [3]:
from itertools import permutations
import math
perms = ["".join(t) for t in permutations("aab")]
print(perms)
print("Không trùng:", sorted(set(perms)))
for n in [3, 8, 12, 20]:
    print(f"{n}! = {math.factorial(n):,}")

['aab', 'aba', 'aab', 'aba', 'baa', 'baa']
Không trùng: ['aab', 'aba', 'baa']
3! = 6
8! = 40,320
12! = 479,001,600
20! = 2,432,902,008,176,640,000


### 2.3. Đếm tần suất bằng `dict` và `collections.Counter`

- `dict`: `count[ch] = count.get(ch, 0) + 1` — `get(ch, 0)` trả `0` nếu `ch` chưa có.
- `Counter(s)`: làm sẵn việc đếm, trả về dict-con `{ký tự: số lần}`.
- `.values()` cho các giá trị (số lần).

In [4]:
from collections import Counter
s = "carerac"
count = {}
for ch in s:
    count[ch] = count.get(ch, 0) + 1
print("dict tự đếm :", count)
print("Counter     :", Counter(s))
print("các số lần  :", list(Counter(s).values()))

dict tự đếm : {'c': 2, 'a': 2, 'r': 2, 'e': 1}
Counter     : Counter({'c': 2, 'a': 2, 'r': 2, 'e': 1})
các số lần  : [2, 2, 2, 1]


### 2.4. Số lẻ / chẵn: `x % 2`

`x % 2 == 1` ⇔ `x` lẻ. `sum(1 for v in values if v % 2 == 1)` đếm có bao nhiêu giá trị lẻ.

In [5]:
values = [2, 2, 2, 1]
print([v % 2 for v in values])
print("số giá trị lẻ:", sum(1 for v in values if v % 2 == 1))

[0, 0, 0, 1]
số giá trị lẻ: 1


## 3. Cách 1 — Brute force: thử mọi hoán vị

Ý nghĩ ngây thơ: sinh hết hoán vị, xem có cái nào đối xứng không. `any(...)` trả `True` ngay khi gặp phần tử đúng đầu tiên.

Chỉ dùng được cho chuỗi **rất ngắn** (dưới ~9 ký tự). Với `n = 5000` thì `5000!` là con số không tưởng.

In [6]:
def can_permute_brute(s: str) -> bool:
    return any("".join(p) == "".join(p)[::-1] for p in permutations(s))

for s, expected in examples:
    print(s, "->", can_permute_brute(s), "| mong đợi", expected)

code -> False | mong đợi False
aab -> True | mong đợi True
carerac -> True | mong đợi True


In [7]:
import time
t0 = time.perf_counter()
can_permute_brute("abcdefgh")        # 8! = 40320 hoán vị, không cái nào đối xứng
t1 = time.perf_counter()
print(f"8 ký tự khác nhau: {t1 - t0:.3f}s — mỗi ký tự thêm nhân thời gian lên ~n lần")

8 ký tự khác nhau: 0.024s — mỗi ký tự thêm nhân thời gian lên ~n lần


## 4. Insight — palindrome nhìn từ góc độ **tần suất**

Gập palindrome làm đôi: ký tự ở vị trí `i` phải bằng ký tự ở vị trí `n-1-i`. Vậy các ký tự đi **thành cặp**.

```text
r a c e c a r
r-----------r   cặp
  a-------a     cặp
    c---c       cặp
      e         ở giữa (một mình)
```

- Độ dài **chẵn**: mọi ký tự xuất hiện **chẵn** lần.
- Độ dài **lẻ**: đúng **một** ký tự (ở giữa) xuất hiện lẻ lần.

Gộp lại: **số ký tự có tần suất lẻ `<= 1`**.

**Chiều ngược** (đủ điều kiện ⇒ xếp được): nếu tối đa một ký tự lẻ, ta đặt mỗi cặp vào hai đầu đối xứng, ký tự lẻ (nếu có) đặt một bản ở giữa → luôn tạo được palindrome. Vậy điều kiện là **cần và đủ**.

In [8]:
for s in ["code", "aab", "carerac", "aabb", "abc"]:
    c = Counter(s)
    odd = [ch for ch, v in c.items() if v % 2 == 1]
    print(f"{s!r:>10}: tần suất = {dict(c)}, ký tự lẻ = {odd} -> {len(odd) <= 1}")

    'code': tần suất = {'c': 1, 'o': 1, 'd': 1, 'e': 1}, ký tự lẻ = ['c', 'o', 'd', 'e'] -> False
     'aab': tần suất = {'a': 2, 'b': 1}, ký tự lẻ = ['b'] -> True
 'carerac': tần suất = {'c': 2, 'a': 2, 'r': 2, 'e': 1}, ký tự lẻ = ['e'] -> True
    'aabb': tần suất = {'a': 2, 'b': 2}, ký tự lẻ = [] -> True
     'abc': tần suất = {'a': 1, 'b': 1, 'c': 1}, ký tự lẻ = ['a', 'b', 'c'] -> False


### 4.1. Cách 2 — Áp dụng insight: sắp xếp rồi đếm từng nhóm

Sau `sorted(s)`, các ký tự giống nhau đứng liền nhau thành **nhóm**. Đếm độ dài mỗi nhóm, đếm nhóm lẻ. Không cần dict, nhưng tốn `O(n log n)` cho sắp xếp.

In [9]:
def can_permute_sort(s: str) -> bool:
    t = sorted(s)
    odd = 0
    i = 0
    while i < len(t):
        j = i
        while j < len(t) and t[j] == t[i]:   # mở rộng nhóm ký tự giống t[i]
            j += 1
        if (j - i) % 2 == 1:
            odd += 1
        i = j
    return odd <= 1

print(sorted("carerac"))
print([can_permute_sort(s) for s, _ in examples])

['a', 'a', 'c', 'c', 'e', 'r', 'r']
[False, True, True]


## 5. Cách 3 — Lời giải chính: đếm bằng `Counter`, đếm số lẻ

**Đoạn 1:** `count = Counter(s)` — tần suất từng ký tự, `O(n)`.

**Đoạn 2:** `odd = sum(1 for v in count.values() if v % 2 == 1)` — có bao nhiêu ký tự lẻ.

**Đoạn 3:** `return odd <= 1`.

In [10]:
def can_permute_counter(s: str) -> bool:
    count = Counter(s)
    odd = sum(1 for v in count.values() if v % 2 == 1)
    return odd <= 1

for s, expected in examples:
    assert can_permute_counter(s) == expected
print([can_permute_counter(s) for s, _ in examples])

[False, True, True]


## 6. Cách 4 — `set` "bật/tắt" trong một lượt duyệt

Quan sát: ta chỉ cần **tính chẵn lẻ**, không cần số lần. Dùng một `set` `odd_chars` chứa các ký tự **đang lẻ**:
- gặp `ch` mà `ch` đã trong set → bây giờ chẵn → `remove`.
- chưa có → bây giờ lẻ → `add`.

Cuối cùng `len(odd_chars) <= 1`.

In [11]:
def can_permute_set(s: str) -> bool:
    odd_chars = set()
    for ch in s:
        if ch in odd_chars:
            odd_chars.remove(ch)
        else:
            odd_chars.add(ch)
    return len(odd_chars) <= 1

print([can_permute_set(s) for s, _ in examples])

[False, True, True]


## 7. TRACE từng bước (cách set) với `s = "carerac"`

| i | ch | trong set trước? | hành động | odd_chars sau |
|---:|:---:|:---:|---|---|
| 0 | c | không | add | {c} |
| 1 | a | không | add | {c, a} |
| 2 | r | không | add | {c, a, r} |
| 3 | e | không | add | {c, a, r, e} |
| 4 | r | có | remove | {c, a, e} |
| 5 | a | có | remove | {c, e} |
| 6 | c | có | remove | {e} |

`len = 1 <= 1` → `True` (`e` là ký tự ở giữa `racecar`).

In [12]:
def trace_can_permute(s: str) -> bool:
    odd_chars = set()
    print(f"s = {s!r}")
    for i, ch in enumerate(s):          # enumerate cho cả chỉ số i và ký tự ch
        before = ch in odd_chars
        if before:
            odd_chars.remove(ch)
            action = "remove (giờ chẵn)"
        else:
            odd_chars.add(ch)
            action = "add    (giờ lẻ) "
        print(f"  i={i} ch={ch!r} | có sẵn? {str(before):<5} | {action} | odd = {sorted(odd_chars)}")
    print(f"  -> số ký tự lẻ = {len(odd_chars)} -> {len(odd_chars) <= 1}")
    return len(odd_chars) <= 1

trace_can_permute("carerac")
print()
trace_can_permute("code")

s = 'carerac'
  i=0 ch='c' | có sẵn? False | add    (giờ lẻ)  | odd = ['c']
  i=1 ch='a' | có sẵn? False | add    (giờ lẻ)  | odd = ['a', 'c']
  i=2 ch='r' | có sẵn? False | add    (giờ lẻ)  | odd = ['a', 'c', 'r']
  i=3 ch='e' | có sẵn? False | add    (giờ lẻ)  | odd = ['a', 'c', 'e', 'r']
  i=4 ch='r' | có sẵn? True  | remove (giờ chẵn) | odd = ['a', 'c', 'e']
  i=5 ch='a' | có sẵn? True  | remove (giờ chẵn) | odd = ['c', 'e']
  i=6 ch='c' | có sẵn? True  | remove (giờ chẵn) | odd = ['e']
  -> số ký tự lẻ = 1 -> True

s = 'code'
  i=0 ch='c' | có sẵn? False | add    (giờ lẻ)  | odd = ['c']
  i=1 ch='o' | có sẵn? False | add    (giờ lẻ)  | odd = ['c', 'o']
  i=2 ch='d' | có sẵn? False | add    (giờ lẻ)  | odd = ['c', 'd', 'o']
  i=3 ch='e' | có sẵn? False | add    (giờ lẻ)  | odd = ['c', 'd', 'e', 'o']
  -> số ký tự lẻ = 4 -> False


False

## 8. Cách 5 — Bitmask XOR (26 bit)

Vì chỉ có chữ thường `a..z`, gán ký tự thứ `k` (`k = ord(ch) - ord('a')`) cho **bit thứ k** của một số nguyên `mask`.

- `mask ^= 1 << k` **lật** bit `k` (XOR với 1 đổi 0↔1) — đúng là "bật/tắt" như set.
- Cuối cùng `mask` có bit 1 ở các ký tự lẻ. Cần **tối đa một bit 1**:
  - `mask == 0`, hoặc
  - `mask & (mask - 1) == 0` (phép này xóa bit 1 thấp nhất; nếu ra 0 thì chỉ có một bit 1).

Các phép mới:
- `ord(ch)` → mã số của ký tự (`ord('a') = 97`).
- `1 << k` → số có đúng bit `k` bật (`1 << 3 = 8 = 0b1000`).
- `^` XOR, `&` AND theo bit; `bin(x)` in dạng nhị phân.

In [13]:
print("ord('a') =", ord("a"), "| ord('e') - ord('a') =", ord("e") - ord("a"))
print("1 << 3 =", 1 << 3, bin(1 << 3))
m = 0
m ^= 1 << 2; print("lật bit 2:", bin(m))
m ^= 1 << 2; print("lật lại  :", bin(m))
for x in [0, 8, 12, 16, 7]:
    print(f"x = {x:>2} = {bin(x):>7} | x & (x-1) = {bin(x & (x - 1)) if x else '0b0'} -> tối đa 1 bit 1? {x & (x - 1) == 0}")

ord('a') = 97 | ord('e') - ord('a') = 4
1 << 3 = 8 0b1000
lật bit 2: 0b100
lật lại  : 0b0
x =  0 =     0b0 | x & (x-1) = 0b0 -> tối đa 1 bit 1? True
x =  8 =  0b1000 | x & (x-1) = 0b0 -> tối đa 1 bit 1? True
x = 12 =  0b1100 | x & (x-1) = 0b1000 -> tối đa 1 bit 1? False
x = 16 = 0b10000 | x & (x-1) = 0b0 -> tối đa 1 bit 1? True
x =  7 =   0b111 | x & (x-1) = 0b110 -> tối đa 1 bit 1? False


In [14]:
def can_permute_bitmask(s: str) -> bool:
    mask = 0
    for ch in s:
        mask ^= 1 << (ord(ch) - ord("a"))
    return mask & (mask - 1) == 0

def trace_bitmask(s):
    mask = 0
    for ch in s:
        mask ^= 1 << (ord(ch) - ord("a"))
        print(f"  {ch}: mask = {mask:026b}")
    print("  kết quả:", mask & (mask - 1) == 0)

print([can_permute_bitmask(s) for s, _ in examples])
print("(bit của 'a' nằm ở ngoài cùng bên PHẢI, 'b' kế bên trái, ...)")
trace_bitmask("aab")

[False, True, True]
(bit của 'a' nằm ở ngoài cùng bên PHẢI, 'b' kế bên trái, ...)
  a: mask = 00000000000000000000000001
  a: mask = 00000000000000000000000000
  b: mask = 00000000000000000000000010
  kết quả: True


Chú ý `mask = 0`: `0 & (0 - 1) = 0 & -1 = 0` → `True`, nên công thức gộp luôn trường hợp "không ký tự lẻ".

## 9. Code nộp LeetCode

### 9.1. Bản Counter (dễ đọc — khuyên dùng)

In [15]:
from collections import Counter

class Solution:
    def canPermutePalindrome(self, s: str) -> bool:
        odd = sum(1 for v in Counter(s).values() if v % 2 == 1)
        return odd <= 1

sol = Solution()
print([sol.canPermutePalindrome(s) for s in ["code", "aab", "carerac"]])

[False, True, True]


### 9.2. Bản set một lượt và bản bitmask (đổi tên class thành `Solution` khi nộp)

In [16]:
class SolutionSet:
    def canPermutePalindrome(self, s: str) -> bool:
        odd_chars = set()
        for ch in s:
            if ch in odd_chars:
                odd_chars.remove(ch)
            else:
                odd_chars.add(ch)
        return len(odd_chars) <= 1

class SolutionBitmask:
    def canPermutePalindrome(self, s: str) -> bool:
        mask = 0
        for ch in s:
            mask ^= 1 << (ord(ch) - ord("a"))
        return mask & (mask - 1) == 0

print([SolutionSet().canPermutePalindrome(s) for s in ["code", "aab", "carerac"]])
print([SolutionBitmask().canPermutePalindrome(s) for s in ["code", "aab", "carerac"]])

[False, True, True]
[False, True, True]


## 10. Vì sao đúng? (invariant) + Độ phức tạp

**Invariant cách set/bitmask:** sau khi xử lý `s[0..i]`, `odd_chars` (hay các bit 1 của `mask`) **đúng bằng** tập ký tự xuất hiện **lẻ** lần trong `s[0..i]`. Bước `i+1` chỉ đổi tính chẵn lẻ của đúng `s[i+1]`, và add/remove (hay XOR) làm đúng việc đó.

Kết hợp với định lý ở mục 4 (*hoán vị palindrome tồn tại ⇔ ≤ 1 ký tự lẻ*) ⇒ kết quả đúng.

Gọi `n = len(s)`, `σ` = số ký tự khác nhau (`<= 26`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Thử mọi hoán vị | `O(n! · n)` | `O(n)` | chỉ để minh họa |
| 2. Sắp xếp + đếm nhóm | `O(n log n)` | `O(n)` (list sắp xếp) | không cần hash |
| 3. Counter đếm lẻ | `O(n)` | `O(σ)` | dễ đọc nhất |
| 4. Set bật/tắt | `O(n)` | `O(σ)` | 1 lượt, không cần đếm |
| 5. Bitmask XOR | `O(n)` | `O(1)` (1 số nguyên) | phụ thuộc bảng chữ cái nhỏ |

Với chữ thường, `σ <= 26` nên `O(σ)` cũng là `O(1)`.

## 11. Bộ test

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<36} PASS")

sol = Solution()
check("mẫu: code", sol.canPermutePalindrome("code"), False)
check("mẫu: aab", sol.canPermutePalindrome("aab"), True)
check("mẫu: carerac", sol.canPermutePalindrome("carerac"), True)
check("1 ký tự", sol.canPermutePalindrome("a"), True)
check("2 ký tự khác", sol.canPermutePalindrome("ab"), False)
check("chẵn, mọi tần suất chẵn", sol.canPermutePalindrome("aabbcc"), True)
check("2 ký tự lẻ (phản ví dụ)", sol.canPermutePalindrome("aabbcd"), False)
check("một ký tự lặp 3 lần + cặp", sol.canPermutePalindrome("aaabb"), True)
check("dài 5000 toàn 'a'", sol.canPermutePalindrome("a" * 5000), True)
check("26 chữ cái mỗi chữ 1 lần", sol.canPermutePalindrome("abcdefghijklmnopqrstuvwxyz"), False)

mẫu: code                            PASS
mẫu: aab                             PASS
mẫu: carerac                         PASS
1 ký tự                              PASS
2 ký tự khác                         PASS
chẵn, mọi tần suất chẵn              PASS
2 ký tự lẻ (phản ví dụ)              PASS
một ký tự lặp 3 lần + cặp            PASS
dài 5000 toàn 'a'                    PASS
26 chữ cái mỗi chữ 1 lần             PASS


In [18]:
# Đối chiếu tất cả cách: brute force trên chuỗi ngắn, các cách nhanh trên chuỗi dài ngẫu nhiên
import random
rng = random.Random(266)
fast = [can_permute_sort, can_permute_counter, can_permute_set, can_permute_bitmask,
        Solution().canPermutePalindrome, SolutionSet().canPermutePalindrome, SolutionBitmask().canPermutePalindrome]
for _ in range(400):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 7)))
    expected = can_permute_brute(s)
    assert all(f(s) == expected for f in fast), s
for _ in range(2000):
    s = "".join(rng.choice("abcdefghijklmnopqrstuvwxyz"[:rng.randint(1, 26)]) for _ in range(rng.randint(1, 300)))
    assert len({f(s) for f in fast}) == 1, s
print("Mọi cách khớp brute force và khớp nhau")

Mọi cách khớp brute force và khớp nhau


## 12. Lỗi người mới hay gặp

### Lỗi 1: Yêu cầu **mọi** tần suất chẵn

In [19]:
def wrong_all_even(s):
    return all(v % 2 == 0 for v in Counter(s).values())

print("wrong_all_even('aab') =", wrong_all_even("aab"), "| đúng =", can_permute_counter("aab"))

wrong_all_even('aab') = False | đúng = True


**Vì sao sai:** palindrome độ dài lẻ có một ký tự ở giữa. **Sửa:** cho phép `<= 1` ký tự lẻ.

### Lỗi 2: Kiểm tra chính `s` có phải palindrome

In [20]:
def wrong_check_s(s):
    return s == s[::-1]

print("wrong_check_s('aab') =", wrong_check_s("aab"), "| đúng =", can_permute_counter("aab"))
print("wrong_check_s('carerac') =", wrong_check_s("carerac"), "(đúng một cách may mắn vì 'carerac' vốn đã đối xứng)")

wrong_check_s('aab') = False | đúng = True
wrong_check_s('carerac') = True (đúng một cách may mắn vì 'carerac' vốn đã đối xứng)


**Vì sao sai:** đề hỏi về **hoán vị**, ta được sắp xếp lại tùy ý.

### Lỗi 3: So sánh `odd == 1` thay vì `odd <= 1`

In [21]:
def wrong_eq_one(s):
    return sum(v % 2 for v in Counter(s).values()) == 1

print("wrong_eq_one('aabb') =", wrong_eq_one("aabb"), "| đúng =", can_permute_counter("aabb"))

wrong_eq_one('aabb') = False | đúng = True


### Lỗi 4: Bitmask quên trường hợp `mask == 0` hoặc dùng `|` thay vì `^`

- `mask |= bit` chỉ **bật**, không tắt được → mất thông tin chẵn lẻ.
- Kiểm tra `bin(mask).count("1") == 1` quên trường hợp 0 bit.

In [22]:
def wrong_or(s):
    mask = 0
    for ch in s:
        mask |= 1 << (ord(ch) - ord("a"))      # SAI: OR không lật bit
    return mask & (mask - 1) == 0

print("wrong_or('aabb') =", wrong_or("aabb"), "| đúng =", can_permute_bitmask("aabb"))

wrong_or('aabb') = False | đúng = True


### Lỗi 5: Bitmask với ký tự ngoài `a..z`

Nếu đề mở rộng (chữ hoa, chữ số, dấu cách), `ord(ch) - ord('a')` có thể âm → `1 << âm` gây `ValueError`. Khi bảng chữ cái không cố định, dùng `set`/`Counter`.

In [23]:
try:
    1 << (ord("A") - ord("a"))
except ValueError as e:
    print("ValueError:", e)

ValueError: negative shift count


## 13. Mẹo nhận dạng pattern

- **"Có thể sắp xếp lại / hoán vị thành …"** → thứ tự không quan trọng, chỉ **tần suất** quan trọng → `Counter`.
- **"Palindrome"** + **"tần suất"** → điều kiện "tối đa một lẻ".
- Chỉ cần **chẵn/lẻ** → set bật/tắt hoặc XOR bitmask.
- Bảng chữ cái nhỏ cố định (26 chữ thường) → mảng 26 phần tử hoặc bitmask 26 bit thay cho dict.

Bài liên quan: 409 Longest Palindrome, 267 Palindrome Permutation II, 242 Valid Anagram, 1177 Can Make Palindrome from Substring (prefix XOR), 1915 Number of Wonderful Substrings.

## 14. Biến thể

### 14.1. Longest Palindrome (LeetCode 409): độ dài palindrome dài nhất xây được

Lấy mọi cặp (`v // 2 * 2`), cộng thêm 1 nếu có ký tự lẻ (đặt ở giữa).

In [24]:
def longest_palindrome(s: str) -> int:
    c = Counter(s)
    pairs = sum(v // 2 * 2 for v in c.values())
    has_odd = any(v % 2 == 1 for v in c.values())
    return pairs + (1 if has_odd else 0)

print(longest_palindrome("abccccdd"), longest_palindrome("a"), longest_palindrome("aabb"))

7 1 4


### 14.2. Dựng **một** palindrome từ hoán vị (khi được)

Nửa trái = mỗi ký tự lặp `v // 2` lần; giữa = ký tự lẻ (nếu có); nửa phải = nửa trái đảo ngược.

In [25]:
def build_palindrome(s: str):
    c = Counter(s)
    odd = [ch for ch, v in c.items() if v % 2 == 1]
    if len(odd) > 1:
        return None
    half = "".join(ch * (v // 2) for ch, v in sorted(c.items()))
    mid = odd[0] if odd else ""
    return half + mid + half[::-1]

for s in ["carerac", "aabb", "code", "a"]:
    p = build_palindrome(s)
    print(s, "->", p, "| hợp lệ:", p is None or (p == p[::-1] and sorted(p) == sorted(s)))

carerac -> acrerca | hợp lệ: True
aabb -> abba | hợp lệ: True
code -> None | hợp lệ: True
a -> a | hợp lệ: True


### 14.3. Truy vấn trên chuỗi con bằng prefix XOR

Câu hỏi: đoạn `s[l..r]` có hoán vị palindrome không? Lưu `pre[i]` = XOR mask của `s[0..i-1]`. Mask của đoạn là `pre[r+1] ^ pre[l]` → trả lời mỗi truy vấn `O(1)`.

In [26]:
def build_prefix(s):
    pre = [0]
    for ch in s:
        pre.append(pre[-1] ^ (1 << (ord(ch) - ord("a"))))
    return pre

def query(pre, l, r):
    m = pre[r + 1] ^ pre[l]
    return m & (m - 1) == 0

s = "abacaba"
pre = build_prefix(s)
for l, r in [(0, 6), (0, 1), (1, 3), (2, 4)]:
    print(f"s[{l}..{r}] = {s[l:r+1]!r} -> {query(pre, l, r)} (kiểm: {can_permute_counter(s[l:r+1])})")

s[0..6] = 'abacaba' -> True (kiểm: True)
s[0..1] = 'ab' -> False (kiểm: False)
s[1..3] = 'bac' -> False (kiểm: False)
s[2..4] = 'aca' -> True (kiểm: True)


## 15. Bài tập (tự làm — cell chỉ định nghĩa, không gọi)

**Bài 1 — Dự đoán:** Không chạy code, chuỗi nào sau đây có hoán vị palindrome: `"tactcoa"`, `"aabbccd"`, `"abab"`, `"abcabcxy"`? Điền list bool.

In [27]:
# TODO: điền dự đoán theo thứ tự ["tactcoa", "aabbccd", "abab", "abcabcxy"]
predict = None
# assert predict == [Solution().canPermutePalindrome(s) for s in ["tactcoa", "aabbccd", "abab", "abcabcxy"]]

**Bài 2 — Điền chỗ trống: dùng mảng 26 phần tử thay cho dict.**

In [28]:
def exercise_array26(s: str) -> bool:
    freq = [0] * 26
    # TODO 1: tăng freq[ord(ch) - ord('a')] cho mỗi ch
    # TODO 2: đếm số phần tử lẻ trong freq
    # TODO 3: trả về số lẻ <= 1
    pass

# assert [exercise_array26(s) for s in ["code", "aab", "carerac"]] == [False, True, True]

**Bài 3 — Tự viết lại bản bitmask** không nhìn mục 8.

In [29]:
def exercise_bitmask(s: str) -> bool:
    # TODO
    pass

# assert [exercise_bitmask(s) for s in ["code", "aab", "carerac", "a"]] == [False, True, True, True]

**Bài 4 — Biến thể không phân biệt hoa/thường, bỏ qua ký tự không phải chữ cái** (ví dụ `"Tact Coa"` → `True`). Gợi ý: `ch.isalpha()`, `ch.lower()`.

In [30]:
def exercise_phrase(s: str) -> bool:
    # TODO
    pass

# assert exercise_phrase("Tact Coa") == True and exercise_phrase("Hello, World") == False

**Bài 5 — Số ký tự tối thiểu cần xóa** để phần còn lại có hoán vị palindrome.

In [31]:
def exercise_min_delete(s: str) -> int:
    # TODO
    pass

# assert exercise_min_delete("code") == 3 and exercise_min_delete("aab") == 0 and exercise_min_delete("aabbcd") == 1

**Bài 6 — Mini challenge: đếm số chuỗi con** (liên tiếp, khác vị trí) có hoán vị palindrome, `O(n · 26)`. Gợi ý: prefix XOR + đếm số lần gặp mỗi mask.

In [32]:
def exercise_count_substrings(s: str) -> int:
    # TODO
    pass

# assert exercise_count_substrings("aba") == 4      # "a","b","a","aba"
# assert exercise_count_substrings("aabb") == 9

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | đếm ký tự lẻ trong từng chuỗi | `tactcoa`: t2 a2 c2 o1 | lẻ `<= 1` → True |
| 2 | `freq` thay cho `Counter` | `freq[ord(ch)-97] += 1` | `sum(f % 2 for f in freq) <= 1` |
| 3 | lật bit bằng `^=` | `1 << (ord(ch) - ord('a'))` | `mask & (mask - 1) == 0` |
| 4 | lọc rồi dùng lại hàm cũ | `[ch.lower() for ch in s if ch.isalpha()]` | gọi `can_permute_counter` |
| 5 | mỗi ký tự lẻ "thừa" một bản | giữ lại được 1 ký tự lẻ | `max(0, odd - 1)` |
| 6 | mask đoạn = `pre[r+1] ^ pre[l]` | đoạn hợp lệ khi mask có ≤ 1 bit | với mỗi mask, cộng `seen[mask]` và `seen[mask ^ (1<<k)]` |

## 17. Đáp án tham khảo

In [33]:
answer_predict = [can_permute_counter(s) for s in ["tactcoa", "aabbccd", "abab", "abcabcxy"]]
print("Bài 1:", answer_predict, "(abcabcxy có x và y lẻ -> False)")

def answer_array26(s: str) -> bool:
    freq = [0] * 26
    for ch in s:
        freq[ord(ch) - ord("a")] += 1
    return sum(f % 2 for f in freq) <= 1

def answer_bitmask(s: str) -> bool:
    mask = 0
    for ch in s:
        mask ^= 1 << (ord(ch) - ord("a"))
    return mask & (mask - 1) == 0

def answer_phrase(s: str) -> bool:
    letters = [ch.lower() for ch in s if ch.isalpha()]
    return sum(v % 2 for v in Counter(letters).values()) <= 1

def answer_min_delete(s: str) -> int:
    odd = sum(v % 2 for v in Counter(s).values())
    return max(0, odd - 1)

def answer_count_substrings(s: str) -> int:
    seen = {0: 1}
    mask = 0
    total = 0
    for ch in s:
        mask ^= 1 << (ord(ch) - ord("a"))
        total += seen.get(mask, 0)                   # đoạn có mask 0 (mọi tần suất chẵn)
        for k in range(26):
            total += seen.get(mask ^ (1 << k), 0)    # đoạn có đúng 1 bit lẻ
        seen[mask] = seen.get(mask, 0) + 1
    return total

def brute_count_substrings(s):
    return sum(can_permute_counter(s[i:j]) for i in range(len(s)) for j in range(i + 1, len(s) + 1))

assert answer_predict == [True, True, True, False]
assert [answer_array26(s) for s in ["code", "aab", "carerac"]] == [False, True, True]
assert [answer_bitmask(s) for s in ["code", "aab", "carerac", "a"]] == [False, True, True, True]
assert answer_phrase("Tact Coa") == True and answer_phrase("Hello, World") == False
assert answer_min_delete("code") == 3 and answer_min_delete("aab") == 0 and answer_min_delete("aabbcd") == 1
assert answer_count_substrings("aba") == 4 and answer_count_substrings("aabb") == 9
rng = random.Random(2660)
for _ in range(200):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 12)))
    assert answer_count_substrings(s) == brute_count_substrings(s), s
print("Tất cả đáp án PASS")

Bài 1: [True, True, True, False] (abcabcxy có x và y lẻ -> False)
Tất cả đáp án PASS


## 18. Template nhớ nhanh + checklist submit

```python
from collections import Counter
class Solution:
    def canPermutePalindrome(self, s: str) -> bool:
        return sum(v % 2 for v in Counter(s).values()) <= 1
```

**Câu thần chú:** *"Palindrome = các cặp + tối đa một ký tự lẻ ở giữa."*

**Checklist:**
- [ ] Điều kiện là `<= 1` ký tự lẻ (không phải `== 0`, không phải `== 1`).
- [ ] Không kiểm tra `s == s[::-1]` — đề hỏi về hoán vị.
- [ ] Bitmask: dùng `^` (lật), kiểm `mask & (mask - 1) == 0`.
- [ ] Nhớ `from collections import Counter` nếu dùng Counter.
- [ ] Tên method `canPermutePalindrome`, trả `bool`.